In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
import re
pd.set_option("display.width", 250)              # widen the line
pd.set_option("display.expand_frame_repr", False)  # never wrap, scroll sideways

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs
from helpers.check_model_performance import review_typical_slices
run_against_models = import_module("model_baseline.01_process_dumb_models").run_against_models
_libpostal = import_module("model_baseline.04_dumb_libpostal_model")   # digit-prefixed filename -> can't use `from ... import`
_get_libpostal_results = _libpostal._get_libpostal_results
_vocab_step = import_module("siamese_model.01_vocab_step")   # digit-prefixed filename -> can't use `from ... import`
normalize, build_vocab = _vocab_step.normalize, _vocab_step.build_vocab
PAD_ID, UNK_ID = _vocab_step.PAD_ID, _vocab_step.UNK_ID


In [2]:
ot_train, ot_train_label = load_pairs("overture", "train")
ot_val, ot_val_label = load_pairs("overture", "val")
ot_test, ot_test_label = load_pairs("overture", "test")
ot_all = pd.concat([ot_train, ot_val, ot_test])

st_train, st_train_label = load_pairs("stamford", "train")
st_val, st_val_label = load_pairs("stamford", "val")
st_test, st_test_label = load_pairs("stamford", "test")
st_all = pd.concat([st_train, st_val, st_test])

## normalize
`normalize` and `build_vocab` live in `01_vocab_step.py`; this notebook only tests them.

In [3]:
# test normalize function
assert normalize(f"     {'a'*20}   {'a'*10}  ") == f"{'a'*20} {'a'*10}"   # collapses whitespace, strips the ends
assert normalize(f"{'A' * 20}") == "a" * 20                              # lowercases
assert len(normalize(f"{'A'*110}")) == 85                                # truncates to 85
assert normalize("  12  MAIN\nST ") == "12 main st"                      # newline counts as whitespace
assert normalize(" " * 10 + "a" * 90) == "a" * 85                        # strip happens before the cut
print("normalize: all passed")

normalize: all passed


## build_vocab

In [4]:
vocab = build_vocab(ot_train, min_count = 1)
print(len(vocab), vocab)

40 {'<pad>': 0, '<unk>': 1, ' ': 2, 'e': 3, 't': 4, 'a': 5, 's': 6, 'r': 7, 'n': 8, 'o': 9, 'i': 10, 'd': 11, '1': 12, 'l': 13, '2': 14, '0': 15, 'm': 16, '5': 17, 'v': 18, '3': 19, 'w': 20, '4': 21, 'h': 22, 'p': 23, '6': 24, 'c': 25, '7': 26, '9': 27, '8': 28, 'b': 29, 'u': 30, 'g': 31, 'k': 32, 'y': 33, 'f': 34, 'x': 35, 'q': 36, '-': 37, 'j': 38, 'z': 39}


In [5]:
# test build_vocab
all_chars = set("".join(ot_train["address_a"].map(normalize)) + "".join(ot_train["address_b"].map(normalize)))

assert vocab["<pad>"] == PAD_ID and vocab["<unk>"] == UNK_ID
assert len(vocab) == len(all_chars) + 2                    # every character seen + the two special ids
assert all(vocab[c] >= 2 for c in all_chars)               # real characters never collide with pad/unk
assert sorted(vocab.values()) == list(range(len(vocab)))   # ids are 0..n-1, no gaps or duplicates
assert vocab.get("%", UNK_ID) == UNK_ID                    # unseen character -> unk
assert build_vocab(ot_train, min_count = 1) == vocab       # deterministic on the same data
assert build_vocab(ot_train.sample(frac = 1, random_state = 0), min_count = 1) == vocab   # ...and independent of row order

# min_count: a character that appears once disappears at min_count=2
toy = pd.DataFrame({"address_a": ["12 main st", "14 main st"], "address_b": ["12 main street", "9 elm st#"]})
toy_vocab = build_vocab(toy, min_count = 2)
assert "#" not in toy_vocab and toy_vocab.get("#", UNK_ID) == UNK_ID
assert "m" in toy_vocab
print("build_vocab: all passed")

build_vocab: all passed


In [6]:
# with the real cutoff (20), does anything drop out of the Overture vocab?
vocab_20 = build_vocab(ot_train)
print(len(vocab_20), "ids at min_count=20 vs", len(vocab), "at min_count=1; dropped:", set(vocab) - set(vocab_20))

40 ids at min_count=20 vs 40 at min_count=1; dropped: set()


## Stamford vs Overture characters
Stamford's side B is character-corrupted, so its character set may differ. This decides how a shared vocab works (training-data row).

In [7]:
st_vocab = build_vocab(st_train, min_count = 1)
print(len(st_vocab), st_vocab)
print("in Stamford, not Overture:", set(st_vocab) - set(vocab))
print("in Overture, not Stamford:", set(vocab) - set(st_vocab))

39 {'<pad>': 0, '<unk>': 1, ' ': 2, 'r': 3, 'e': 4, 'a': 5, 'd': 6, 'o': 7, 'l': 8, 'n': 9, 't': 10, 's': 11, 'i': 12, '1': 13, 'v': 14, '2': 15, 'c': 16, 'h': 17, 'w': 18, '3': 19, '4': 20, 'g': 21, '5': 22, 'u': 23, 'b': 24, '6': 25, 'p': 26, 'm': 27, '7': 28, '0': 29, '8': 30, 'k': 31, '9': 32, 'y': 33, 'f': 34, 'x': 35, 'j': 36, 'z': 37, 'q': 38}
in Stamford, not Overture: set()
in Overture, not Stamford: {'-'}
